# Comprehensive Model Lab Experiments

This notebook follows [EXPERIMENT_PLAN.md](../EXPERIMENT_PLAN.md) and [EXPERIMENT_PLAN_2.md](../EXPERIMENT_PLAN_2.md).

**What we are doing (plain English):**

1. Load recent Nobitex USDT_IRT 30-second intervals from Postgres.
2. For each model, simulate how production learns online: predict step *t* with what you knew at *t−1*, then update.
3. Score every model by **mean negative log-likelihood (NLL)** — lower is better.
4. Compare **model families** (tails, joint flow, state-conditioned flow, full-book price vs mid-only, GARCH, HMM, …), not just hyperparameter grids.

**How to read results:**

| Column | Meaning |
|--------|--------|
| `track` | Which research question (A1 tails, B3 GARCH, …) |
| `mean_nll` | Average surprise per step — **lower wins** within the same family |
| `vs_baseline` | Paired improvement vs production baseline (negative = candidate wins) |
| `cold→warm` | First-half minus second-half NLL (positive = model improves as it learns) |

Flow models and price models are **not comparable** to each other.

In [2]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from mm.env_loader import load_mm_env
from research.model_lab.core.analysis import (
    dataset_profile,
    leaderboard,
    paired_nll_deltas,
    paired_significance,
    period_split_nll,
    rolling_mean_nll,
    stationarity_report,
    stratified_nll,
    stratified_nll_quintiles,
)
from research.model_lab.core.walk_forward import run_walk_forward
from research.model_lab.data.loader import load_interval_steps, parse_since
from research.model_lab.models.registry import build_model

load_mm_env()

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

OUTPUT_DIR = REPO_ROOT / "research" / "model_lab" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"repo: {REPO_ROOT}")

ModuleNotFoundError: No module named 'mm'

In [ ]:
DATA_CONFIG = {
    "exchange": "nobitex",
    "symbol": "USDT_IRT",
    "interval_ms": 30_000,
    "since": "48h",
    "source": "auto",
}
MIN_UPDATES_BEFORE_SCORE = 1
DATA_CONFIG

## Step 1 — Understand the data

Before comparing models, we profile sparsity, volume tails, and price drift. This tells us *why* hurdle flow models and regime-aware price models matter.

In [ ]:
until = datetime.now(timezone.utc)
since = parse_since(DATA_CONFIG["since"], until=until)
steps = load_interval_steps(
    exchange=DATA_CONFIG["exchange"],
    symbol=DATA_CONFIG["symbol"],
    interval_ms=DATA_CONFIG["interval_ms"],
    since=since,
    until=until,
    source=DATA_CONFIG["source"],
)
profile = dataset_profile(steps)
print(json.dumps(profile, indent=2, default=str))
assert len(steps) >= 2, "Need at least 2 interval steps"

In [ ]:
def spec_from(kind: str, **config) -> str:
    return kind if not config else f"{kind}{json.dumps(config, separators=(',', ':'))}"


def short_label(spec: str, max_len: int = 48) -> str:
    return spec if len(spec) <= max_len else spec[: max_len - 3] + "..."


def evaluate_track(track_specs: dict[str, str], *, steps=steps):
    results = {}
    for track, spec in track_specs.items():
        model = build_model(spec)
        result = run_walk_forward(model, steps, min_updates_before_score=MIN_UPDATES_BEFORE_SCORE)
        results[track] = (spec, result)
        summary = result.summary
        print(f"[{track:12s}] mean_nll={summary.mean_nll:.4f}  scored={summary.n_scored}")
    return results


def track_table(results: dict[str, tuple[str, object]], *, baseline_spec: str, family: str):
    by_spec = {spec: result for _, (spec, result) in results.items()}
    board = leaderboard(by_spec)
    pairwise = {row["label"]: row for row in paired_nll_deltas(by_spec, baseline_label=baseline_spec)}
    rows = []
    for track, (spec, result) in sorted(results.items()):
        summary = result.summary
        split = period_split_nll(result)
        pair = pairwise.get(spec, {})
        rows.append(
            {
                "track": track,
                "model": short_label(spec),
                "mean_nll": summary.mean_nll,
                "median_nll": summary.median_nll,
                "vs_baseline": pair.get("mean_delta_nll"),
                "win_rate": pair.get("win_rate"),
                "cold_to_warm": split.get("warmup_improvement"),
            }
        )
    if pd is not None:
        return pd.DataFrame(rows).sort_values("mean_nll")
    return sorted(rows, key=lambda r: r["mean_nll"] or 1e18)


def display_df(df):
    if pd is not None:
        display(df)
    else:
        print(df)

## Track A — Flow models (`V_buy`, `V_sell`)

| Track | Question |
|-------|----------|
| **A0** | Trivial baseline (always predict zero volume) |
| **A1** | Production hurdle lognormal vs other tail families |
| **A2** | Independent sides vs joint buy/sell pattern (bivariate hurdle) |
| **A3** | Heavy tails (log-Student-t) |
| **A4** | State-conditioned gate (spread + book imbalance) |

Production baseline: `flow:hurdle_lognormal` with decay=0.95.

In [ ]:
FLOW_BASELINE = spec_from("flow:hurdle_lognormal", decay=0.95, init_p0=0.3, init_mu=0.0, init_sigma=1.0)

FLOW_TRACKS = {
    "A0_zero": "baseline:zero_flow",
    "A1_prod": FLOW_BASELINE,
    "A1_gamma": spec_from("flow:hurdle_gamma", decay=0.95, init_p0=0.3),
    "A1_weibull": spec_from("flow:hurdle_weibull", decay=0.95, init_p0=0.3),
    "A1_gamma_plain": spec_from("flow:gamma", decay=0.95),
    "A1_weibull_plain": spec_from("flow:weibull", decay=0.95),
    "A2_bivariate": spec_from("flow:bivariate_hurdle_lognormal", decay=0.95, init_mu=0.0, init_sigma=1.0),
    "A3_student_t": spec_from("flow:hurdle_log_student_t", decay=0.95, init_p0=0.3, init_mu=0.0, init_sigma=1.0, init_df=3.0),
    "A4_state_gate": spec_from("flow:state_conditioned_hurdle", decay=0.95, init_p0=0.3, init_mu=0.0, init_sigma=1.0),
}

print("Running flow track experiments...")
flow_results = evaluate_track(FLOW_TRACKS)
flow_table = track_table(flow_results, baseline_spec=FLOW_BASELINE, family="flow")
display_df(flow_table)

In [ ]:
flow_by_spec = {spec: result for _, (spec, result) in flow_results.items()}
flow_regimes = stratified_nll(flow_by_spec, steps)
if pd is not None:
    display_df(pd.DataFrame(flow_regimes).sort_values(["volume_regime", "mean_nll"]))
else:
    print(flow_regimes[:20])

## Track B — Price models (book transitions)

| Track | Question |
|-------|----------|
| **B0** | Simple random walk on mid |
| **B1** | Full-book production models (conditional mixture + residual) |
| **B2** | Mid-only targets with flow features |
| **B3** | GARCH volatility clustering |
| **B4** | AR(1) autocorrelation on mid deltas |
| **B5** | 2-state HMM (quiet vs active) |
| **B6** | Microprice and spread-normalized targets |

Production baseline: `price:conditional_mechanical_mixture` (decay=0.99, ref_vol=1000).

In [ ]:
PRICE_BASELINE = spec_from(
    "price:conditional_mechanical_mixture",
    decay=0.99,
    reference_vwap_vol=1000.0,
    shape_alpha=0.5,
    student_df=3.0,
    initial_sigma=50.0,
    initial_log_sigma=0.25,
)

PRICE_TRACKS = {
    "B0_mid_rw": "baseline:mid_random_walk",
    "B1_prod_mix": PRICE_BASELINE,
    "B1_residual": spec_from("price:residual", residual_decay=0.95, residual_sigma=50.0, reference_vwap_vol=1000.0),
    "B2_student_t": spec_from("price:student_t_mid_delta", decay=0.99, initial_sigma=50.0, df=3.0),
    "B2_flow_cond": spec_from("price:flow_conditioned_mid_delta", decay=0.99, variance_decay=0.99, initial_sigma=50.0),
    "B3_garch": spec_from("price:garch_mid_delta", omega=1.0, alpha=0.05, beta=0.90, initial_sigma=50.0),
    "B4_ar1": spec_from("price:ar1_mid_delta", decay=0.99, phi_decay=0.99, initial_sigma=50.0),
    "B5_hmm2": spec_from("price:hmm2_mid_delta", decay=0.99, initial_sigma_quiet=30.0, initial_sigma_active=80.0),
    "B6_microprice": spec_from("price:microprice_delta", decay=0.99, initial_sigma=50.0),
    "B6_spread_norm": spec_from("price:spread_norm_mid_delta", decay=0.99, initial_sigma=1.0),
}

print("Running price track experiments...")
price_results = evaluate_track(PRICE_TRACKS)
price_table = track_table(price_results, baseline_spec=PRICE_BASELINE, family="price")
display_df(price_table)

In [ ]:
price_by_spec = {spec: result for _, (spec, result) in price_results.items()}
price_quintiles = stratified_nll_quintiles(price_by_spec, steps)
if pd is not None:
    display_df(pd.DataFrame(price_quintiles).sort_values(["volume_quintile", "mean_nll"]))
else:
    print(price_quintiles[:20])

## Track C — Stationarity diagnostics

If transformed series (log returns, spread-normalized moves) are stable across time blocks, fixed-weight or neural models become viable. High block-to-block drift means adaptive EWMA/RLS is still the right default.

In [ ]:
stationarity = stationarity_report(steps, blocks=8)
if pd is not None:
    display_df(pd.DataFrame(stationarity))
else:
    print(stationarity)

if plt is not None and stationarity:
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot([r["block"] for r in stationarity], [r["spread_norm_delta_std"] for r in stationarity], "o-", label="spread_norm_delta_std")
    ax.plot([r["block"] for r in stationarity], [r["log_return_std"] for r in stationarity], "o-", label="log_return_std")
    ax.set_xlabel("time block")
    ax.set_ylabel("std")
    ax.legend()
    ax.set_title("Stationarity: stable log-return std but drifting spread-norm std → prefer adaptive models")
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()

## Statistical significance (top candidates vs production)

Bootstrap 95% CI on paired per-step NLL differences. If the entire CI is below zero, the candidate reliably beats the baseline.

In [ ]:
def best_track(results, *, skip_prefixes=("A0", "B0")):
    candidates = [(t, s, r) for t, (s, r) in results.items() if not any(t.startswith(p) for p in skip_prefixes)]
    candidates.sort(key=lambda x: x[2].summary.mean_nll or 1e18)
    return candidates[0]


best_flow_track, best_flow_spec, best_flow_result = best_track(flow_results)
best_price_track, best_price_spec, best_price_result = best_track(price_results)

flow_sig = paired_significance(flow_by_spec, baseline_label=FLOW_BASELINE, candidate_label=best_flow_spec)
price_sig = paired_significance(price_by_spec, baseline_label=PRICE_BASELINE, candidate_label=best_price_spec)

print("Flow significance:", json.dumps(flow_sig, indent=2, default=str))
print("Price significance:", json.dumps(price_sig, indent=2, default=str))

## Verdict — plain English summary

The cell below writes a short report you can share. It names the winner per family, whether production is beaten, and caveats about quiet vs active regimes.

In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")


def winner_line(results, baseline_spec, family_name):
    best_track_id, best_spec, best_res = best_track(results)
    prod_track = next(t for t, (s, _) in results.items() if s == baseline_spec)
    prod_nll = results[prod_track][1].summary.mean_nll
    best_nll = best_res.summary.mean_nll
    beats_prod = best_nll < prod_nll if best_spec != baseline_spec else False
    return (
        f"**{family_name} winner:** `{best_track_id}` ({short_label(best_spec)}) — mean NLL {best_nll:.4f}"
        f" (production: {prod_nll:.4f}, {'beats' if beats_prod else 'does not beat'} production)"
    )


report_lines = [
    f"# Comprehensive Experiment Report ({run_id})",
    "",
    f"**Data:** {profile['n_steps']} steps, {profile['zero_volume_fraction']:.1%} zero-volume, "
    f"mid {profile['mid_min']:.0f}–{profile['mid_max']:.0f} IRT",
    "",
    "## Flow (Track A)",
    winner_line(flow_results, FLOW_BASELINE, "Flow"),
    "",
    "Key questions answered:",
    "- **A1 tails:** compare hurdle_lognormal vs gamma/weibull families in the flow table above.",
    "- **A2 joint:** bivariate hurdle models (0,0)/(>,0) pattern explicitly.",
    "- **A3/A4:** log-Student-t tails and state-conditioned gates.",
    "",
    "## Price (Track B)",
    winner_line(price_results, PRICE_BASELINE, "Price"),
    "",
    "Key questions answered:",
    "- **B1:** full-book mixture/residual vs simpler mid-only models (B2–B6).",
    "- **B3–B5:** GARCH, AR(1), and 2-state HMM mid models.",
    "- **B6:** microprice and spread-normalized stationary targets.",
    "",
    "## Stationarity (Track C)",
    "- Inspect block std table: if spread_norm_delta_std drifts, defer neural sequence models.",
    "",
    "## Significance",
    f"- Flow best vs prod: ΔNLL={flow_sig.get('mean_delta_nll', 'n/a')}, CI=[{flow_sig.get('ci95_low')}, {flow_sig.get('ci95_high')}]",
    f"- Price best vs prod: ΔNLL={price_sig.get('mean_delta_nll', 'n/a')}, CI=[{price_sig.get('ci95_low')}, {price_sig.get('ci95_high')}]",
    "",
    "## Caveats",
    "- Check volume-regime tables: a quiet-regime winner may not help burst risk.",
    "- Mid-only price models use a different observation space than full-book mixture.",
    "- NLL improvement does not guarantee better MPC rollouts — shadow-test before deploy.",
]

report_md = "\n".join(report_lines)
report_path = OUTPUT_DIR / f"{run_id}_comprehensive_plan_REPORT.md"
latest_path = OUTPUT_DIR / "LATEST_comprehensive_plan_REPORT.md"
report_path.write_text(report_md, encoding="utf-8")
latest_path.write_text(report_md, encoding="utf-8")

from IPython.display import Markdown, display

display(Markdown(report_md))
print(f"\nSaved: {report_path}")